# Sign Language Translator - Standalone Google Colab Notebook

This notebook provides a complete end-to-end workflow for training a Sign Language (ASL) alphabet classifier using PyTorch. **This is a standalone version that contains all the necessary code in one file.**

## What you'll learn:
- Computer Vision basics with PyTorch
- Handling image datasets (Sign Language MNIST)
- Using Transfer Learning with pretrained models (ResNet, MobileNet, EfficientNet)
- Training and evaluating a deep learning model
- Visualizing results and running inference

**GPU Recommendation**: For faster training, enable GPU in Runtime > Change runtime type > Hardware accelerator > T4 GPU (or any available GPU).

## 1. Imports and Core Logic

First, we'll import the necessary libraries and define our core functions for data handling, model building, and training.

In [ ]:
import os
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, Literal, Dict, List, Any

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms as T

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
### --- DATASET UTILITIES ---

LABEL_MAP = {
    # Kaggle Sign Language MNIST uses 24 letters (no J, Z static)
    # The labels range from 0 to 24, skipping 9 (J)
    0: "A", 1: "B", 2: "C", 3: "D", 4: "E", 5: "F",
    6: "G", 7: "H", 8: "I", 10: "K", 11: "L", 12: "M",
    13: "N", 14: "O", 15: "P", 16: "Q", 17: "R", 18: "S",
    19: "T", 20: "U", 21: "V", 22: "W", 23: "X", 24: "Y",
}

def save_images_from_csv(csv_path: Path, out_dir: Path, split: str) -> None:
    """Converts Kaggle CSV format to PNG images in an ImageFolder structure."""
    df = pd.read_csv(csv_path)
    has_label = "label" in df.columns
    pixel_cols = [c for c in df.columns if c.startswith("pixel")]

    for idx, row in df.iterrows():
        if has_label:
            label_num = int(row["label"])
            label = LABEL_MAP.get(label_num, str(label_num))
            class_dir = out_dir / split / label
        else:
            class_dir = out_dir / split / "unknown"
        
        class_dir.mkdir(parents=True, exist_ok=True)
        pixels = row[pixel_cols].to_numpy(dtype=np.uint8)
        img = pixels.reshape(28, 28)
        pil = Image.fromarray(img, mode="L")
        pil.save(class_dir / f"{idx:06d}.png")

### --- TRANSFORMS ---

def build_train_transforms(image_size: Tuple[int, int] = (224, 224), grayscale: bool = False):
    """Data augmentation for training."""
    transform_list = [
        T.Resize(image_size),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2) if not grayscale else T.Lambda(lambda x: x),
        T.ToTensor(),
    ]
    if grayscale:
        transform_list.insert(0, T.Grayscale(num_output_channels=1))
    # Remove the placeholder Lambda if it was added
    transform_list = [t for t in transform_list if not (isinstance(t, T.Lambda) and not grayscale)]
    return T.Compose(transform_list)

def build_eval_transforms(image_size: Tuple[int, int] = (224, 224), grayscale: bool = False):
    """Standard preprocessing for evaluation/inference."""
    transform_list = [
        T.Resize(image_size),
        T.ToTensor(),
    ]
    if grayscale:
        transform_list.insert(0, T.Grayscale(num_output_channels=1))
    return T.Compose(transform_list)

### --- MODELS ---

BackboneName = Literal["resnet18", "resnet34", "mobilenet_v3_small", "mobilenet_v3_large", "efficientnet_b0"]

@dataclass
class ModelConfig:
    backbone: BackboneName = "resnet18"
    num_classes: int = 24
    pretrained: bool = True
    dropout: Optional[float] = 0.2
    in_channels: int = 1

class RepeatTo3Channels(nn.Module):
    """Helper to convert 1-channel grayscale to 3-channel for pretrained models."""
    def __init__(self, in_channels: int) -> None:
        super().__init__()
        self.conv = None if in_channels == 3 else nn.Conv2d(in_channels, 3, kernel_size=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor: 
        return self.conv(x) if self.conv is not None else x

def build_classifier(config: ModelConfig) -> nn.Module:
    """Builds a classifier with a pretrained backbone."""
    weights = "DEFAULT" if config.pretrained else None
    
    if config.backbone == "resnet18":
        model = models.resnet18(weights=weights)
        in_feats = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(config.dropout) if config.dropout else nn.Identity(), nn.Linear(in_feats, config.num_classes))
    elif config.backbone == "resnet34":
        model = models.resnet34(weights=weights)
        in_feats = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(config.dropout) if config.dropout else nn.Identity(), nn.Linear(in_feats, config.num_classes))
    elif "mobilenet" in config.backbone:
        model = getattr(models, config.backbone)(weights=weights)
        in_feats = model.classifier[-1].in_features
        model.classifier[-1] = nn.Sequential(nn.Dropout(config.dropout) if config.dropout else nn.Identity(), nn.Linear(in_feats, config.num_classes))
    elif "efficientnet" in config.backbone:
        model = getattr(models, config.backbone)(weights=weights)
        in_feats = model.classifier[-1].in_features
        model.classifier[-1] = nn.Sequential(nn.Dropout(config.dropout) if config.dropout else nn.Identity(), nn.Linear(in_feats, config.num_classes))
    
    if config.in_channels != 3:
        stem = RepeatTo3Channels(config.in_channels)
        model = nn.Sequential(stem, model)
    
    return model

### --- TRAINING & UTILS ---

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    losses, accs = [], []
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        accs.append((outputs.argmax(1) == targets).float().mean().item())
    return np.mean(losses), np.mean(accs)

def evaluate(model, loader, criterion, device):
    model.eval()
    losses, accs = [], []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            losses.append(loss.item())
            accs.append((outputs.argmax(1) == targets).float().mean().item())
    return np.mean(losses), np.mean(accs)

print("✓ Core logic loaded successfully!")

## 2. Download Dataset from Kaggle

We'll use the **Sign Language MNIST** dataset. 

1. **Get Kaggle API credentials**:
   - Go to [Kaggle Account](https://www.kaggle.com/account)
   - Click "Create New API Token" to download `kaggle.json`
   - Upload the file when prompted below.

In [ ]:
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✓ Kaggle credentials configured!")
else:
    print("✓ Kaggle credentials already exist.")

In [ ]:
# Download and extract dataset
!kaggle datasets download -d datamunge/sign-language-mnist
!unzip -q sign-language-mnist.zip -d raw_data
print("✓ Dataset downloaded and extracted!")

## 3. Data Preparation

The dataset is in CSV format. We will convert it into an image folder structure: `data/train/<LABEL>/000001.png`

In [ ]:
# Convert CSV to images
data_dir = Path("data")
save_images_from_csv(Path("raw_data/sign_mnist_train.csv"), data_dir, "train")
save_images_from_csv(Path("raw_data/sign_mnist_test.csv"), data_dir, "test")

train_dir = data_dir / "train"
test_dir = data_dir / "test"

classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
print(f"✓ Data conversion complete! Found {len(classes)} classes.")
print(f"Classes: {', '.join(classes)}")

## 4. Visualize Data

Let's see what the hand signs look like.

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(15, 6))
fig.suptitle('Sample ASL Hand Signs (Sign MNIST)', fontsize=16)

for idx, cls in enumerate(classes):
    if idx >= 24: break
    ax = axes.flat[idx]
    img_path = random.choice(list((train_dir / cls).glob("*.png")))
    img = Image.open(img_path)
    ax.imshow(img, cmap='gray')
    ax.set_title(cls)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Training

Now we'll set up the data loaders and train our model.

In [ ]:
# Configuration
CONFIG = {
    "backbone": "resnet18",
    "epochs": 10,
    "batch_size": 64,
    "lr": 1e-3,
    "image_size": (224, 224),
    "grayscale": True,
    "seed": 42
}

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Datasets & Loaders
train_tfms = build_train_transforms(CONFIG["image_size"], CONFIG["grayscale"])
test_tfms = build_eval_transforms(CONFIG["image_size"], CONFIG["grayscale"])

train_ds = datasets.ImageFolder(root=str(train_dir), transform=train_tfms)
test_ds = datasets.ImageFolder(root=str(test_dir), transform=test_tfms)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2)

# Model
model = build_classifier(ModelConfig(
    backbone=CONFIG["backbone"],
    num_classes=len(classes),
    in_channels=1 if CONFIG["grayscale"] else 3
))
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

print(f"Training {CONFIG['backbone']} on {device}...")

# History for plotting
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(1, CONFIG["epochs"] + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    ts_loss, ts_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()
    
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["test_loss"].append(ts_loss)
    history["test_acc"].append(ts_acc)
    
    print(f"Epoch {epoch:02d}/{CONFIG['epochs']} | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Test Loss: {ts_loss:.4f} Acc: {ts_acc:.4f}")

print("\n✓ Training complete!")

## 6. Evaluation

Let's visualize the training progress and the model's confusion matrix.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train")
plt.plot(history["test_loss"], label="Test")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_acc"], label="Train")
plt.plot(history["test_acc"], label="Test")
plt.title("Accuracy")
plt.legend()
plt.show()

In [ ]:
# Get all predictions for confusion matrix
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs.to(device))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(targets.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## 7. Random Inference

Let's see the model in action on random images from the test set.

In [ ]:
num_samples = 12
fig, axes = plt.subplots(3, 4, figsize=(15, 12))

model.eval()
for i, ax in enumerate(axes.flat):
    # Pick a random class and image
    cls = random.choice(classes)
    img_path = random.choice(list((test_dir / cls).glob("*.png")))
    img = Image.open(img_path)
    
    # Predict
    input_tensor = test_tfms(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = output.argmax(1).item()
        prob = torch.softmax(output, dim=1)[0, pred_idx].item()
    
    pred_cls = classes[pred_idx]
    
    ax.imshow(img, cmap='gray')
    color = 'green' if pred_cls == cls else 'red'
    ax.set_title(f"True: {cls}\nPred: {pred_cls} ({prob:.1%})", color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 8. Export Model

Finally, we save the model for future use.

In [ ]:
save_path = "sign_language_model.pt"
torch.save({
    "model_state": model.state_dict(),
    "classes": classes,
    "config": CONFIG
}, save_path)

print(f"✓ Model saved to {save_path}")

# Download to local machine
files.download(save_path)

---
**Congratulations!** You've built a complete Sign Language Translator in a single notebook. 🚀